# Space Debris Sensitivity Analysis — Model Exploration

This notebook walks through both debris evolution models interactively before running the full sensitivity analysis.

**Models:**
- `CascadeModel` — deterministic Kessler-type population ODE
- `MultiShellModel` — four-shell stochastic MOCAT-inspired model

**Primary sensitivity variable:** Solar flux F10.7 (affects atmospheric drag → debris lifetime)

---

In [ ]:
import sys
sys.path.insert(0, '..')  # allow imports from project root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

from models.cascade_model import CascadeModel
from models.multishell_model import MultiShellModel, SHELL_MIDS_KM

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Cascade Model — Baseline (F10.7 = 150, PMD = 90%)

In [ ]:
c_base = CascadeModel()
r_base = c_base.run(years=200)

print(f"tau_D at baseline F10.7=150: {c_base.tau_D():.1f} yr")
print(f"tau_S_eff at PMD=90%: {c_base.tau_S_eff():.1f} yr")
print(f"Initial collision rate: {c_base.alpha_SS*c_base.S0**2 + c_base.alpha_SD*c_base.S0*c_base.D0:.3f} /yr")
print(f"Final N (200 yr): {r_base['N'][-1]:,.0f}")
print(f"Final collision rate: {r_base['R_col'][-1]:.3f} /yr")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(r_base['t'], r_base['S'], label='Intact (S)', color='steelblue')
axes[0].plot(r_base['t'], r_base['D'], label='Debris (D)', color='firebrick')
axes[0].plot(r_base['t'], r_base['N'], label='Total (N)', color='black', ls='--')
axes[0].set_xlabel('Time [yr]'); axes[0].set_ylabel('Objects')
axes[0].set_title('Population over time'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(r_base['t'], r_base['R_col'], color='darkorange', lw=2)
axes[1].set_xlabel('Time [yr]'); axes[1].set_ylabel('Collisions / yr')
axes[1].set_title('Collision rate'); axes[1].grid(alpha=0.3)

# Phase portrait: D vs S
axes[2].plot(r_base['S'], r_base['D'], color='purple', lw=1.5)
axes[2].scatter([r_base['S'][0]], [r_base['D'][0]], color='green', zorder=5, label='t=0')
axes[2].scatter([r_base['S'][-1]], [r_base['D'][-1]], color='red', zorder=5, label='t=200yr')
axes[2].set_xlabel('Intact objects S'); axes[2].set_ylabel('Debris D')
axes[2].set_title('Phase portrait'); axes[2].legend(); axes[2].grid(alpha=0.3)

fig.suptitle('Cascade model — baseline (F10.7=150, PMD=90%)', fontsize=13)
plt.tight_layout()

## 2. Effect of Solar Flux on the Cascade Model

In [ ]:
f107_values = [70, 100, 130, 150, 180, 200, 230]
colors = plt.cm.RdYlBu_r(np.linspace(0, 1, len(f107_values)))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for f107, col in zip(f107_values, colors):
    c = CascadeModel(F10_7=f107)
    r = c.run(years=200)
    label = f'F10.7={f107}'
    axes[0].plot(r['t'], r['N'], color=col, label=label, lw=1.5)
    axes[1].plot(r['t'], r['D'], color=col, label=label, lw=1.5)

for ax, title, ylabel in [
    (axes[0], 'Total population N', 'Total objects'),
    (axes[1], 'Debris D only', 'Debris fragments'),
]:
    ax.set_xlabel('Time [yr]'); ax.set_ylabel(ylabel)
    ax.set_title(title); ax.legend(fontsize=8); ax.grid(alpha=0.3)

fig.suptitle('Cascade model sensitivity to F10.7', fontsize=13)
plt.tight_layout()

## 3. Multi-Shell Model — Baseline

In [ ]:
ms_base = MultiShellModel(seed=42)
ms_ens = ms_base.run_ensemble(years=200, n_runs=20)

t = ms_ens['t']
mean_N = ms_ens['N_total_mean']
std_N = ms_ens['N_total_std']

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# All runs + mean
ax = axes[0]
for run in ms_ens['N_total_all']:
    ax.plot(t, run, color='firebrick', alpha=0.15, lw=0.8)
ax.plot(t, mean_N, color='firebrick', lw=2.5, label='Ensemble mean')
ax.fill_between(t, mean_N - std_N, mean_N + std_N, color='firebrick', alpha=0.2, label='±1σ')
ax.set_xlabel('Time [yr]'); ax.set_ylabel('Total objects (all shells)')
ax.set_title('Multi-shell: ensemble trajectories'); ax.legend(); ax.grid(alpha=0.3)

# Single run: per-shell breakdown
ms_single = MultiShellModel(seed=0)
r_single = ms_single.run(years=200)
ax = axes[1]
shell_colors = ['#1a6b9a', '#2e9b5e', '#d4831b', '#b02020']
for i, (mid, col) in enumerate(zip(SHELL_MIDS_KM, shell_colors)):
    ax.plot(r_single['t'], r_single['N'][:, i], color=col, lw=1.5, label=f'{mid} km')
ax.set_xlabel('Time [yr]'); ax.set_ylabel('Objects per shell')
ax.set_title('Multi-shell: per-altitude-band populations'); ax.legend(); ax.grid(alpha=0.3)

fig.suptitle('Multi-shell model — baseline (F10.7=150, PMD=90%)', fontsize=13)
plt.tight_layout()

## 4. Effect of Solar Flux on the Multi-Shell Model

In [ ]:
from models.multishell_model import debris_lifetime

# Show how debris lifetime varies with F10.7 across shells
f107_range = np.linspace(70, 230, 100)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ax = axes[0]
shell_colors = ['#1a6b9a', '#2e9b5e', '#d4831b', '#b02020']
for i, (mid, col) in enumerate(zip(SHELL_MIDS_KM, shell_colors)):
    lifetimes = [debris_lifetime(f)[i] for f in f107_range]
    ax.plot(f107_range, lifetimes, color=col, lw=2, label=f'{mid} km')
ax.set_xlabel('F10.7 [sfu]'); ax.set_ylabel('Debris lifetime τ_D [yr]')
ax.set_title('Debris lifetime vs solar flux by shell')
ax.legend(); ax.grid(alpha=0.3); ax.set_yscale('log')

# Multi-shell F10.7 comparison
ax = axes[1]
for f107, col in zip([70, 130, 150, 200, 230], plt.cm.RdYlBu_r(np.linspace(0, 1, 5))):
    ms = MultiShellModel(F10_7=f107, seed=99)
    ens = ms.run_ensemble(years=200, n_runs=15)
    ax.plot(ens['t'], ens['N_total_mean'], color=col, lw=2, label=f'F10.7={f107}')
ax.set_xlabel('Time [yr]'); ax.set_ylabel('Total N (all shells)')
ax.set_title('Multi-shell sensitivity to F10.7')
ax.legend(fontsize=9); ax.grid(alpha=0.3)

fig.suptitle('Multi-shell model — solar flux sensitivity', fontsize=13)
plt.tight_layout()

## 5. Side-by-Side Comparison: Both Models at Solar Min vs Solar Max

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

for ax, f107, linestyle, label_suffix in [
    (axes[0], 70, '-', 'Solar min (F10.7=70)'),
    (axes[0], 230, '--', 'Solar max (F10.7=230)'),
    (axes[1], 70, '-', 'Solar min (F10.7=70)'),
    (axes[1], 230, '--', 'Solar max (F10.7=230)'),
]:
    pass  # handled below

for f107, ls, alpha_v in [(70, '-', 1.0), (230, '--', 1.0)]:
    # Cascade
    r = CascadeModel(F10_7=f107).run(years=200)
    axes[0].plot(r['t'], r['N'], color='steelblue', ls=ls, lw=2,
                 label=f'Cascade F10.7={f107}')

    # Multi-shell
    ens = MultiShellModel(F10_7=f107, seed=0).run_ensemble(years=200, n_runs=20)
    axes[1].plot(ens['t'], ens['N_total_mean'], color='firebrick', ls=ls, lw=2,
                 label=f'Multi-shell F10.7={f107}')

for ax, title in [(axes[0], 'Cascade model'), (axes[1], 'Multi-shell model')]:
    ax.set_xlabel('Time [yr]'); ax.set_ylabel('Total objects N')
    ax.set_title(title); ax.legend(); ax.grid(alpha=0.3)

fig.suptitle('Solar min vs solar max — trajectory comparison', fontsize=13)
plt.tight_layout()

# Print sensitivity ratio
for label, Model, kwargs in [
    ('Cascade', CascadeModel, {}),
    ('Multi-shell', MultiShellModel, {'seed': 0}),
]:
    if label == 'Cascade':
        N_min = Model(F10_7=70, **kwargs).run(years=200)['N'][-1]
        N_max = Model(F10_7=230, **kwargs).run(years=200)['N'][-1]
    else:
        N_min = Model(F10_7=70, **kwargs).run_ensemble(years=200, n_runs=20)['N_total_mean'][-1]
        N_max = Model(F10_7=230, **kwargs).run_ensemble(years=200, n_runs=20)['N_total_mean'][-1]
    print(f'{label:15s}  N(solar min) = {N_min:8,.0f}  N(solar max) = {N_max:8,.0f}  ratio = {N_min/N_max:.2f}')

## 6. Quick Sweep Preview

A small taste of the full sensitivity analysis — F10.7 sweep on final population.

In [ ]:
f107_grid = np.linspace(70, 230, 17)
c_finals = []
ms_finals = []

for f107 in f107_grid:
    c_finals.append(CascadeModel(F10_7=f107).run(years=200)['N'][-1])
    ms_finals.append(MultiShellModel(F10_7=f107, seed=7).run_ensemble(years=200, n_runs=15)['N_total_mean'][-1])

c_finals = np.array(c_finals)
ms_finals = np.array(ms_finals)

# Normalise to F10.7 = 150
ref_idx = np.abs(f107_grid - 150).argmin()
c_norm = c_finals / c_finals[ref_idx]
ms_norm = ms_finals / ms_finals[ref_idx]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(f107_grid, c_norm, color='#2563EB', lw=2.5, label='Cascade (normalised)')
ax.plot(f107_grid, ms_norm, color='#DC2626', lw=2.5, ls='--', label='Multi-shell (normalised)')
ax.axvline(70, color='gray', ls=':', lw=1, label='Solar min')
ax.axvline(230, color='orange', ls=':', lw=1, label='Solar max')
ax.axhline(1.0, color='black', ls=':', lw=0.8, alpha=0.5)
ax.set_xlabel('F10.7 [sfu]')
ax.set_ylabel('Normalised final total population')
ax.set_title('Preview: F10.7 sweep — 200-yr final population (normalised to F10.7=150)')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()

print(f"Cascade    solar-min/max ratio: {c_finals[0]/c_finals[-1]:.3f}")
print(f"Multi-shell solar-min/max ratio: {ms_finals[0]/ms_finals[-1]:.3f}")

---
## Next steps

Run the full analysis from the project root:
```powershell
python run_analysis.py --quick   # fast test
python run_analysis.py           # full analysis
```

Results saved to `results/`.